# v34 master compare and final selector

Run this after any subset of v34 route notebooks. It compares all available route summaries and exports `v34_final_*` only if a route beats v32.2 and passes artifact checks.

In [ ]:
import json, re, math, os, sys, random, itertools, statistics
from pathlib import Path
from collections import Counter, defaultdict

CANDIDATE_DIRS = [
    Path('/kaggle/working'),
    Path('/kaggle/input/datasets/yixuanisthebest/v2333333'),
    Path('/kaggle/input/datasets/nguyenminhtric/test-pipeline'),
    Path('/kaggle/input'),
    Path('/mnt/data'),
]

def find_file(names, required=True):
    if isinstance(names, str): names = [names]
    for d in CANDIDATE_DIRS:
        for name in names:
            p = d / name
            if p.exists(): return p
    for d in CANDIDATE_DIRS:
        if d.exists():
            for root, dirs, files in os.walk(d):
                for name in names:
                    if name in files:
                        return Path(root)/name
    if required: raise FileNotFoundError(f'Could not find any of {names} in {CANDIDATE_DIRS}')
    return None

def load_json(names, required=True):
    p = find_file(names, required=required)
    if p is None: return None, None
    with open(p, 'r', encoding='utf-8') as f: return json.load(f), p

def save_json(obj, name):
    p = Path('/kaggle/working')/name
    try: p.parent.mkdir(parents=True, exist_ok=True)
    except Exception: pass
    with open(p, 'w', encoding='utf-8') as f: json.dump(obj, f, ensure_ascii=False, indent=2)
    print('saved', p)
    return p

LABELS = ['A','B','C','D','Yes','No','Unknown']

def safe_pred(x):
    return x if x in LABELS else 'UNPARSEABLE'

def prf_for_label(rows, lab):
    tp=sum(1 for r in rows if r.get('gold')==lab and r.get('pred')==lab)
    fp=sum(1 for r in rows if r.get('gold')!=lab and r.get('pred')==lab)
    fn=sum(1 for r in rows if r.get('gold')==lab and r.get('pred')!=lab)
    prec=tp/(tp+fp) if tp+fp else 0.0
    rec=tp/(tp+fn) if tp+fn else 0.0
    f1=2*prec*rec/(prec+rec) if prec+rec else 0.0
    return {'precision':prec,'recall':rec,'f1':f1,'support':tp+fn,'pred_count':tp+fp}

def metrics(rows):
    n=len(rows); acc=sum(1 for r in rows if r.get('gold')==r.get('pred'))/n
    per={lab: prf_for_label(rows, lab) for lab in LABELS}
    macro=sum(per[lab]['f1'] for lab in LABELS)/len(LABELS)
    weighted=sum(per[lab]['f1']*per[lab]['support'] for lab in LABELS)/sum(per[lab]['support'] for lab in LABELS)
    mc=[r for r in rows if r.get('is_mc')]
    ynu=[r for r in rows if not r.get('is_mc')]
    mc_labs=['A','B','C','D']; ynu_labs=['Yes','No','Unknown']
    mc_macro=sum(prf_for_label(mc, lab)['f1'] for lab in mc_labs)/4 if mc else 0
    ynu_macro=sum(prf_for_label(ynu, lab)['f1'] for lab in ynu_labs)/3 if ynu else 0
    return {'n':n,'acc':acc,'macro_f1':macro,'weighted_f1':weighted,'mc_macro':mc_macro,'ynu_macro':ynu_macro,'per_label':per,
            'gold_count':dict(Counter(r.get('gold') for r in rows)), 'pred_count':dict(Counter(r.get('pred') for r in rows))}

def diffs(a,b):
    return [r1['idx'] for r1,r2 in zip(a,b) if r1.get('pred')!=r2.get('pred')]

def clone_rows(rows): return [dict(r) for r in rows]

def load_baselines():
    v27, p27 = load_json('v27_standard_preds.json', required=False)
    base, pb = load_json(['v32_2_full_preds.json','v32_2_independent_full_preds.json','v32_2_standard_preds.json','v30_1_full_preds.json'], required=True)
    print('baseline path:', pb)
    base_m = metrics(base)
    print('baseline macro:', base_m['macro_f1'])
    return v27, p27, base, pb, base_m

BANKED = {14,71,109,125}
BASELINE_MACRO = 0.596858548901435

In [ ]:
v27,p27,base,pb,base_m=load_baselines()
route_files=[
    ('cpu', 'v34_cpu_best_summary.json', 'v34_cpu_best_preds.json'),
    ('base_advocate', 'v34_base_advocate_full_summary.json', 'v34_base_advocate_full_preds.json'),
]
routes=[]
for name,sf,pf in route_files:
    s,sp=load_json(sf, required=False); p,pp=load_json(pf, required=False)
    if s and p:
        m=metrics(p)
        routes.append({'route':name,'summary_path':str(sp),'pred_path':str(pp),'selection_status':s.get('selection_status'),
                       'metrics':m,'delta_macro':m['macro_f1']-base_m['macro_f1'], 'preds':p})
print('routes found', [(r['route'], r['selection_status'], r['metrics']['macro_f1'], r['delta_macro']) for r in routes])

In [ ]:
qualified=[]
for r in routes:
    m=r['metrics']; per=m['per_label']; bper=base_m['per_label']
    d=diffs(base,r['preds'])
    mc_ok=abs(m['mc_macro']-base_m['mc_macro'])<1e-12
    unk_ok=per['Unknown']['f1'] >= bper['Unknown']['f1']-1e-12
    no_ok=per['No']['f1'] >= bper['No']['f1']-0.03
    collapse=per['Yes']['pred_count']>45 or per['No']['pred_count']<30
    strict=m['macro_f1'] > base_m['macro_f1'] + 0.01
    if strict and mc_ok and unk_ok and no_ok and not collapse:
        rr=dict(r); rr['diffs_from_v32_2']=d; rr.pop('preds',None); qualified.append(rr)
qualified=sorted(qualified, key=lambda x:x['metrics']['macro_f1'], reverse=True)
if qualified:
    best=qualified[0]
    preds=[r for r in routes if r['route']==best['route']][0]['preds']
    save_json(preds,'v34_final_preds.json')
    reload,_=load_json('v34_final_preds.json')
    rm=metrics(reload)
    assert abs(rm['macro_f1']-best['metrics']['macro_f1'])<1e-12
    summary={'version':'v34_master_final','selection_status':'SELECT_V34','selected_route':best,'metrics':rm,'baseline_v32_2':base_m,
             'all_routes':[ {k:v for k,v in r.items() if k!='preds'} for r in routes]}
    save_json(summary,'v34_final_summary.json')
    print('SELECT V34', best['route'], rm['macro_f1'])
else:
    save_json(base,'v34_final_preds.json')
    summary={'version':'v34_master_final','selection_status':'ROLLBACK_V32_2','reason':'no route passed strict master gates','metrics':base_m,'baseline_v32_2':base_m,
             'all_routes':[ {k:v for k,v in r.items() if k!='preds'} for r in routes]}
    save_json(summary,'v34_final_summary.json')
    print('ROLLBACK FINAL: keep v32.2')